# COMP9501 team runner

Open this notebook with your own authorized GitHub/Google account, or download the notebook from the private repository and upload it to Colab. The source snapshot is embedded; no GitHub token or runtime git clone is needed. GPU allocation and subscription benefits belong to each runtime account. See COLLABORATION.md in the repository.

Run all performs environment setup and offline checks. Training, checkpoint restoration and result export use the explicit switches below. Change the source in a Git branch/PR, then rebuild this launcher for a new snapshot. This runtime folder is an execution copy.


In [ ]:
# Project source, checked before importing. Do not edit the encoded payload.
from pathlib import Path
import base64, hashlib, os, sys, subprocess, uuid
SOURCE_SNAPSHOT = '92620b7ad5c1d2dfb2e665db93aee92e766409f8f42ef23e4047f066fc3ded1f'
_helper_bytes = base64.b64decode('IiIiQ3JlZGVudGlhbC1mcmVlIHNvdXJjZSBpbXBvcnQsIGNoZWNrZWQgY2hlY2twb2ludCByZXN0b3JlIGFuZCBydW4gZXhwb3J0LiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgsUHVyZVBvc2l4UGF0aAppbXBvcnQgaGFzaGxpYixpbyxqc29uLG9zLHN0YXQsemlwZmlsZQoKCmRlZiBzaGEyNTZfZmlsZShwYXRoKToKICAgIGg9aGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBQYXRoKHBhdGgpLm9wZW4oJ3JiJykgYXMgc3RyZWFtOgogICAgICAgIGZvciBibG9jayBpbiBpdGVyKGxhbWJkYTpzdHJlYW0ucmVhZCgxMDI0KjEwMjQpLGInJyk6aC51cGRhdGUoYmxvY2spCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKCmRlZiBfZW50cmllcyhhcmNoaXZlLG1heF9ieXRlcyk6CiAgICBlbnRyaWVzPWFyY2hpdmUuaW5mb2xpc3QoKTtuYW1lcz1baS5maWxlbmFtZSBmb3IgaSBpbiBlbnRyaWVzXQogICAgaWYgbGVuKGVudHJpZXMpPjUwMDAgb3IgbGVuKG5hbWVzKSE9bGVuKHNldChuYW1lcykpOnJhaXNlIFZhbHVlRXJyb3IoJ0R1cGxpY2F0ZSBvciBleGNlc3NpdmUgWklQIGVudHJpZXMnKQogICAgaWYgc3VtKGkuZmlsZV9zaXplIGZvciBpIGluIGVudHJpZXMpPm1heF9ieXRlczpyYWlzZSBWYWx1ZUVycm9yKCdaSVAgZXhjZWVkcyB1bmNvbXByZXNzZWQtc2l6ZSBsaW1pdCcpCiAgICBmb3IgaXRlbSBpbiBlbnRyaWVzOgogICAgICAgIG5hbWU9aXRlbS5maWxlbmFtZTtwYXRoPVB1cmVQb3NpeFBhdGgobmFtZSkKICAgICAgICBpZiBjaHIoOTIpIGluIG5hbWU6cmFpc2UgVmFsdWVFcnJvcignQmFja3NsYXNoIGluIFpJUCBtZW1iZXInKQogICAgICAgIGlmIGNocig5MikgaW4gbmFtZTpyYWlzZSBWYWx1ZUVycm9yKCdCYWNrc2xhc2ggaW4gWklQIG1lbWJlcicpCiAgICAgICAgaWYgaXRlbS5pc19kaXIoKSBvciBwYXRoLmlzX2Fic29sdXRlKCkgb3IgJy4uJyBpbiBwYXRoLnBhcnRzIG9yICdcXCcgaW4gbmFtZSBvciAnOicgaW4gbmFtZSBvciBub3QgcGF0aC5wYXJ0cyBvciBwYXRoLmFzX3Bvc2l4KCkhPW5hbWU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgYXJjaGl2ZSBtZW1iZXInKQogICAgICAgIGlmIHN0YXQuU19JU0xOSyhpdGVtLmV4dGVybmFsX2F0dHI+PjE2KTpyYWlzZSBWYWx1ZUVycm9yKCdTeW1saW5rIGFyY2hpdmUgbWVtYmVyJykKICAgIHJldHVybiBlbnRyaWVzCgoKZGVmIF9yZXN0b3JlKGFyY2hpdmUsZmlsZXMsZGVzdGluYXRpb24sbWF4X2J5dGVzKToKICAgIGRlc3RpbmF0aW9uPVBhdGgoZGVzdGluYXRpb24pLnJlc29sdmUoKTtlbnRyaWVzPV9lbnRyaWVzKGFyY2hpdmUsbWF4X2J5dGVzKQogICAgaWYge2UuZmlsZW5hbWUgZm9yIGUgaW4gZW50cmllc30hPXNldChmaWxlcyk6cmFpc2UgVmFsdWVFcnJvcignQXJjaGl2ZSBmaWxlIGxpc3QgZG9lcyBub3QgbWF0Y2ggbWFuaWZlc3QnKQogICAgIyBWYWxpZGF0ZSBldmVyeXRoaW5nIGJlZm9yZSBjcmVhdGluZyBhIGZpbGUsIGluY2x1ZGluZyBjb2xsaXNpb24gY2hlY2tzLgogICAgZm9yIGl0ZW0gaW4gZW50cmllczoKICAgICAgICBjb250ZW50PWFyY2hpdmUucmVhZChpdGVtKTtleHBlY3RlZD1maWxlc1tpdGVtLmZpbGVuYW1lXQogICAgICAgIGlmIGhhc2hsaWIuc2hhMjU2KGNvbnRlbnQpLmhleGRpZ2VzdCgpIT1leHBlY3RlZDpyYWlzZSBWYWx1ZUVycm9yKCdBcmNoaXZlIG1lbWJlciBjaGVja3N1bSBtaXNtYXRjaCcpCiAgICAgICAgdGFyZ2V0PWRlc3RpbmF0aW9uL2l0ZW0uZmlsZW5hbWUKICAgICAgICBpZiBub3QgdGFyZ2V0LnJlc29sdmUoKS5pc19yZWxhdGl2ZV90byhkZXN0aW5hdGlvbik6cmFpc2UgVmFsdWVFcnJvcignUGF0aCBlc2NhcGVzIGRlc3RpbmF0aW9uJykKICAgICAgICBpZiB0YXJnZXQuZXhpc3RzKCkgYW5kIChub3QgdGFyZ2V0LmlzX2ZpbGUoKSBvciBzaGEyNTZfZmlsZSh0YXJnZXQpIT1leHBlY3RlZCk6CiAgICAgICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcignRXhpc3RpbmcgY29udGVudCBkaWZmZXJzOiAnK2l0ZW0uZmlsZW5hbWUpCiAgICBmb3IgaXRlbSBpbiBlbnRyaWVzOgogICAgICAgIHRhcmdldD1kZXN0aW5hdGlvbi9pdGVtLmZpbGVuYW1lCiAgICAgICAgaWYgbm90IHRhcmdldC5leGlzdHMoKToKICAgICAgICAgICAgdGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSk7dGFyZ2V0LndyaXRlX2J5dGVzKGFyY2hpdmUucmVhZChpdGVtKSkKICAgIHJldHVybiBkZXN0aW5hdGlvbgoKCmRlZiBtYXRlcmlhbGl6ZV9zb3VyY2UocGF5bG9hZCxleHBlY3RlZF9zaGEsZGVzdGluYXRpb24pOgogICAgaWYgaGFzaGxpYi5zaGEyNTYocGF5bG9hZCkuaGV4ZGlnZXN0KCkhPWV4cGVjdGVkX3NoYTpyYWlzZSBWYWx1ZUVycm9yKCdTb3VyY2UgYnVuZGxlIGNoZWNrc3VtIG1pc21hdGNoJykKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKGlvLkJ5dGVzSU8ocGF5bG9hZCkpIGFzIGFyY2hpdmU6CiAgICAgICAgX2VudHJpZXMoYXJjaGl2ZSw2NCoxMDI0KjEwMjQpCiAgICAgICAgbWFuaWZlc3Q9anNvbi5sb2FkcyhhcmNoaXZlLnJlYWQoJ2J1bmRsZS1tYW5pZmVzdC5qc29uJykpCiAgICAgICAgZmlsZXM9bWFuaWZlc3RbJ2ZpbGVzJ107ZmlsZXNbJ2J1bmRsZS1tYW5pZmVzdC5qc29uJ109aGFzaGxpYi5zaGEyNTYoYXJjaGl2ZS5yZWFkKCdidW5kbGUtbWFuaWZlc3QuanNvbicpKS5oZXhkaWdlc3QoKQogICAgICAgIHJldHVybiBfcmVzdG9yZShhcmNoaXZlLGZpbGVzLGRlc3RpbmF0aW9uLDY0KjEwMjQqMTAyNCkKCgpkZWYgcmVzdG9yZV9jaGVja3BvaW50cyhhcmNoaXZlX3BhdGgsbWFuaWZlc3RfcGF0aCxkZXN0aW5hdGlvbik6CiAgICBtYW5pZmVzdD1qc29uLmxvYWRzKFBhdGgobWFuaWZlc3RfcGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpKQogICAgaWYgc2hhMjU2X2ZpbGUoYXJjaGl2ZV9wYXRoKSE9bWFuaWZlc3RbJ3NoYTI1NiddOnJhaXNlIFZhbHVlRXJyb3IoJ0NoZWNrcG9pbnQgWklQIGNoZWNrc3VtIG1pc21hdGNoJykKICAgIGlmIG5vdCBhbGwobmFtZS5zdGFydHN3aXRoKCdvdXRwdXRzLycpIGZvciBuYW1lIGluIG1hbmlmZXN0WydmaWxlcyddKTpyYWlzZSBWYWx1ZUVycm9yKCdDaGVja3BvaW50IG1hbmlmZXN0IG11c3QgY29udGFpbiBvdXRwdXRzIG9ubHknKQogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUoYXJjaGl2ZV9wYXRoKSBhcyBhcmNoaXZlOgogICAgICAgIHJldHVybiBfcmVzdG9yZShhcmNoaXZlLG1hbmlmZXN0WydmaWxlcyddLGRlc3RpbmF0aW9uLDUxMioxMDI0KjEwMjQpCgoKZGVmIGV4cG9ydF9ydW4ocHJvamVjdF9yb290LHJ1bl9kaXJlY3RvcnksZGVzdGluYXRpb24sbWV0YWRhdGEpOgogICAgcm9vdD1QYXRoKHByb2plY3Rfcm9vdCkucmVzb2x2ZSgpO3J1bj1QYXRoKHJ1bl9kaXJlY3RvcnkpLnJlc29sdmUoKTtkZXN0PVBhdGgoZGVzdGluYXRpb24pLnJlc29sdmUoKQogICAgaWYgbm90IHJ1bi5pc19yZWxhdGl2ZV90byhyb290LydvdXRwdXRzJykgb3Igbm90IHJ1bi5pc19kaXIoKTpyYWlzZSBWYWx1ZUVycm9yKCdDaG9vc2UgYW4gZXhpc3Rpbmcgb3V0cHV0cyBydW4gZGlyZWN0b3J5JykKICAgIGlmIGRlc3QuaXNfcmVsYXRpdmVfdG8ocnVuKTpyYWlzZSBWYWx1ZUVycm9yKCdFeHBvcnQgWklQIG11c3QgYmUgb3V0c2lkZSB0aGUgc2VsZWN0ZWQgcnVuJykKICAgIGZpbGVzPXNvcnRlZChwIGZvciBwIGluIHJ1bi5yZ2xvYignKicpIGlmIHAuaXNfZmlsZSgpKQogICAgaWYgbm90IGZpbGVzOnJhaXNlIFZhbHVlRXJyb3IoJ1J1biBkaXJlY3RvcnkgaXMgZW1wdHknKQogICAgaWYgYW55KG5vdCBwLnJlc29sdmUoKS5pc19yZWxhdGl2ZV90byhydW4pIGZvciBwIGluIGZpbGVzKTpyYWlzZSBWYWx1ZUVycm9yKCdTeW1saW5rIGxlYXZlcyBydW4gZGlyZWN0b3J5JykKICAgIHJlY29yZD17KiptZXRhZGF0YSwnZmlsZXMnOntwLnJlbGF0aXZlX3RvKHJvb3QpLmFzX3Bvc2l4KCk6c2hhMjU2X2ZpbGUocCkgZm9yIHAgaW4gZmlsZXN9fQogICAgZGVzdC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShkZXN0LCd4Jyxjb21wcmVzc2lvbj16aXBmaWxlLlpJUF9ERUZMQVRFRCkgYXMgYXJjaGl2ZToKICAgICAgICBmb3IgcCBpbiBmaWxlczoKICAgICAgICAgICAgYXJjaGl2ZS53cml0ZShwLHAucmVsYXRpdmVfdG8ocm9vdCkuYXNfcG9zaXgoKSkKICAgICAgICBhcmNoaXZlLndyaXRlc3RyKCdydW4tbWV0YWRhdGEuanNvbicsanNvbi5kdW1wcyhyZWNvcmQsaW5kZW50PTIpKydcbicpCiAgICByZXR1cm4geydwYXRoJzpzdHIoZGVzdCksJ3NoYTI1Nic6c2hhMjU2X2ZpbGUoZGVzdCksJ2J5dGVzJzpkZXN0LnN0YXQoKS5zdF9zaXplfQo=')
assert hashlib.sha256(_helper_bytes).hexdigest() == '63285657e6c57bdb5da9acb23bb0831068a58594c645b6b02150bf3059be8d1f'
_bootstrap = {}
exec(compile(_helper_bytes, '<verified project bootstrap>', 'exec'), _bootstrap)
ROOT = Path('/content') / ('COMP9501-GP-' + SOURCE_SNAPSHOT[:12])
_bootstrap['materialize_source'](base64.b64decode('...'), SOURCE_SNAPSHOT, ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
print('Project source verified:', SOURCE_SNAPSHOT)
print('Runtime folder:', ROOT)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
checks = [('code', 'test_math.py'), ('code', 'test_protocol.py'), ('code/functional_merge', 'test_maths.py'), ('code/coefficient_merge', 'test_core.py'), ('part1-reproduction-20261025/code', 'test_csa.py'), ('scripts', 'test_colab_bundle.py')]
for folder, pattern in checks:
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', folder, '-p', pattern], check=True)


## Optional: restore the existing 12 adapters

Get the checkpoint ZIP from the group's private file store. Upload it using Colab's Files panel or set the path to your own authorized Drive mount. GitHub collaborator access does not grant Drive access automatically. The archive remains outside Git.


In [ ]:
CHECKPOINT_ARCHIVE = ''  # e.g. path to the uploaded ZIP; leave blank to skip
if CHECKPOINT_ARCHIVE:
    from colab_bundle import restore_checkpoints
    restore_checkpoints(CHECKPOINT_ARCHIVE, ROOT / 'assets/checkpoints-2026-10-25.json', ROOT)
    print('Checkpoint archive and every restored file verified.')


## Optional: train a fresh adapter bank

Select a GPU before enabling training. Choose a unique output directory. Each member keeps their own identity for code commits; MEMBER below is just the runtime record label. Existing source-calibration and exploratory-validation limits still apply.


In [ ]:
MEMBER = 'member'  # use your own short alias
SEED = 42
RUN_TRAINING = False
RUN_NAME = MEMBER + '-bank-s' + str(SEED) + '-' + uuid.uuid4().hex[:8]
RUN_DIR = ROOT / 'outputs' / RUN_NAME
if RUN_TRAINING:
    assert not RUN_DIR.exists()
    subprocess.run([sys.executable, 'code/run_bank.py', '--config', 'configs/protocol-v2-a100.json', '--seed', str(SEED), '--output', str(RUN_DIR)], check=True)


## Optional: evaluate one restored adapter

This uses the original task-specific head and source validation protocol. Restore the checkpoint asset first. Public base model and dataset downloads are still required. Keep RUN_TRAINING disabled if you only want this evaluation.


In [ ]:
RUN_EVALUATION = False
EVAL_TASK = 'sst2'  # sst2, mrpc, rte or cola
if RUN_EVALUATION:
    bank = ROOT / ('outputs/bank-v2-a100-s42' if SEED == 42 else 'outputs/replication-s43-s44/bank-s' + str(SEED))
    adapter = bank / EVAL_TASK / 'adapter'
    assert (adapter / 'adapter_model.safetensors').is_file(), 'Restore the checkpoint asset first.'
    RUN_NAME = MEMBER + '-eval-' + EVAL_TASK + '-s' + str(SEED) + '-' + uuid.uuid4().hex[:8]
    RUN_DIR = ROOT / 'outputs' / RUN_NAME
    RUN_DIR.mkdir(parents=True)
    subprocess.run([sys.executable, 'code/evaluate.py', '--config', 'configs/protocol-v2-a100.json', '--task', EVAL_TASK, '--adapter', str(adapter), '--output', str(RUN_DIR / 'result.json')], check=True)


## Export a completed run

Set RUN_DIR to the completed outputs subdirectory if needed. The ZIP includes a file manifest, member label, source snapshot and seed. Download it before the temporary VM disappears, or set EXPORT_DIRECTORY to a folder in your own authorized Drive mount. Nothing uploads automatically to the group's account.


In [ ]:
EXPORT_RESULTS = False
EXPORT_DIRECTORY = ROOT / 'outputs' / 'exports'
DOWNLOAD_ZIP = True
if EXPORT_RESULTS:
    from colab_bundle import export_run
    metadata = {'member': MEMBER, 'seed': SEED, 'source_snapshot': SOURCE_SNAPSHOT, 'config': 'configs/protocol-v2-a100.json'}
    export = export_run(ROOT, RUN_DIR, EXPORT_DIRECTORY / (RUN_NAME + '.zip'), metadata)
    print('Export SHA-256:', export['sha256'])
    if DOWNLOAD_ZIP:
        from google.colab import files
        files.download(export['path'])
